# Home Credit Default Risk — Phase 3: Preprocessing

**Objective:** Transform raw data into a clean, model-ready format.

Pipeline (strict order):
1. Load all files
2. Drop high-null columns (>70%)
3. Fix DAYS anomalies (365243 → NaN + flag)
4. Fix previous_application anomalies
5. Create binary missingness flags for 40–70% null columns
6. Impute missing values (median numeric, mode categorical)
7. Encode categorical columns (Label Encoding)
8. Cap outliers at 99th percentile
9. Log1p transform skewed numeric columns
10. Preprocess each secondary file
11. Schema-align train and test
12. Save cleaned files for Phase 4

In [1]:
import os, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
})
C0, C1 = '#534AB7', '#D85A30'

DATA_DIR   = '../data'              # <-- update to your CSV folder
OUTPUT_DIR = './preprocessed/' # cleaned files saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES = {
    'application_train'     : 'application_train.csv',
    'application_test'      : 'application_test.csv',
    'bureau'                : 'bureau.csv',
    'bureau_balance'        : 'bureau_balance.csv',
    'previous_application'  : 'previous_application.csv',
    'POS_CASH_balance'      : 'POS_CASH_balance.csv',
    'installments_payments' : 'installments_payments.csv',
    'credit_card_balance'   : 'credit_card_balance.csv',
}

dfs = {}
for name, fname in FILES.items():
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        dfs[name] = pd.read_csv(path)
        print(f'  ✓ {fname:<42} {dfs[name].shape}')

train = dfs['application_train'].copy()
test  = dfs['application_test'].copy()
print(f'\nTrain: {train.shape} | Test: {test.shape}')
# ── Load Phase 2b feature priority list (if available) ──────────────────────
PRIORITY_PATH = './phase2b_outputs/feature_priority.csv'
if os.path.exists(PRIORITY_PATH):
    feat_priority = pd.read_csv(PRIORITY_PATH)
    # Features IV < 0.02 AND null > 40% — safe to drop
    P3_DROP_IV_NULL = feat_priority[
        (feat_priority['IV'].fillna(0) < 0.02) &
        (feat_priority['Null_Pct'].fillna(0) > 0.40)
    ]['Feature'].tolist()
    # Features IV >= 0.10 despite high null — must keep
    P3_KEEP_HIGH_NULL = feat_priority[
        (feat_priority['IV'].fillna(0) >= 0.10) &
        (feat_priority['Null_Pct'].fillna(0) > 0.40)
    ]['Feature'].tolist()
    print(f'Phase 2b priority loaded: {len(feat_priority)} features ranked')
    print(f'  IV-guided drop candidates : {len(P3_DROP_IV_NULL)}')
    print(f'  Keep despite high null    : {len(P3_KEEP_HIGH_NULL)}')
else:
    P3_DROP_IV_NULL  = []
    P3_KEEP_HIGH_NULL = []
    print('Phase 2b outputs not found — using null-threshold-only drop strategy.')


  ✓ application_train.csv                      (307511, 122)
  ✓ application_test.csv                       (48744, 121)
  ✓ bureau.csv                                 (1716428, 17)
  ✓ bureau_balance.csv                         (27299925, 3)
  ✓ previous_application.csv                   (1670214, 37)
  ✓ POS_CASH_balance.csv                       (10001358, 8)
  ✓ installments_payments.csv                  (13605401, 8)
  ✓ credit_card_balance.csv                    (3840312, 23)

Train: (307511, 122) | Test: (48744, 121)
Phase 2b priority loaded: 121 features ranked
  IV-guided drop candidates : 28
  Keep despite high null    : 1


## 1. Drop High-Null Columns (>70% missing)

Columns missing more than 70% of values offer very little signal and hurt imputation quality.
We compute the drop list on **train** and apply it to **test** identically.

In [2]:
NULL_THRESHOLD_DROP = 0.70
PROTECTED = ['TARGET', 'SK_ID_CURR']
ID_COLS   = ['SK_ID_CURR', 'SK_ID_BUREAU', 'SK_ID_PREV']

null_pct_train = train.isnull().mean()
drop_cols = [c for c in null_pct_train[null_pct_train > NULL_THRESHOLD_DROP].index
             if c not in PROTECTED]

# Phase 2b integration: add IV-useless + high-null cols to drop list
drop_cols = list(set(drop_cols + [c for c in P3_DROP_IV_NULL
                                   if c not in PROTECTED and c in train.columns]))

# Phase 2b integration: never drop features with strong IV despite high null
drop_cols = [c for c in drop_cols if c not in P3_KEEP_HIGH_NULL]

print(f'Columns to drop (>{NULL_THRESHOLD_DROP*100:.0f}% null): {len(drop_cols)}')
for c in drop_cols:
    print(f'  {c:<45} null={null_pct_train[c]*100:.1f}%')

train = train.drop(columns=drop_cols)
test = test.drop(columns=[c for c in drop_cols if c in test.columns])

print(f'\nTrain after drop: {train.shape}')
print(f'Test  after drop: {test.shape}')

Columns to drop (>70% null): 28
  LANDAREA_AVG                                  null=59.4%
  FONDKAPREMONT_MODE                            null=68.4%
  FLOORSMIN_AVG                                 null=67.8%
  NONLIVINGAPARTMENTS_MEDI                      null=69.4%
  LANDAREA_MODE                                 null=59.4%
  ELEVATORS_AVG                                 null=53.3%
  YEARS_BUILD_MEDI                              null=66.5%
  ENTRANCES_AVG                                 null=50.3%
  COMMONAREA_AVG                                null=69.9%
  ELEVATORS_MODE                                null=53.3%
  NONLIVINGAPARTMENTS_MODE                      null=69.4%
  FLOORSMIN_MEDI                                null=67.8%
  OWN_CAR_AGE                                   null=66.0%
  NONLIVINGAREA_MEDI                            null=55.2%
  YEARS_BUILD_AVG                               null=66.5%
  COMMONAREA_MEDI                               null=69.9%
  LIVINGAPARTMENTS_MEDI 

## 2. Fix DAYS Anomalies

Strategy: Replace with `NaN` + create binary flag `DAYS_EMPLOYED_ANOM`.
The flag itself carries predictive signal — keep it as a feature.

In [3]:
def fix_days_anomalies(df, label=''):
    if 'DAYS_EMPLOYED' not in df.columns:
        return df
    n_anom = (df['DAYS_EMPLOYED'] == 365243).sum()
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    print(f'  [{label}] DAYS_EMPLOYED anomalies replaced: {n_anom:,} ({n_anom/len(df)*100:.2f}%)')
    return df

train = fix_days_anomalies(train, 'train')
test  = fix_days_anomalies(test,  'test')

print(f'\n  DAYS_EMPLOYED after fix:')
print(f'  min={train["DAYS_EMPLOYED"].min():.0f}  max={train["DAYS_EMPLOYED"].max():.0f}  '
      f'null%={train["DAYS_EMPLOYED"].isnull().mean()*100:.2f}%')
print(f'  DAYS_EMPLOYED_ANOM counts: {train["DAYS_EMPLOYED_ANOM"].value_counts().to_dict()}')

  [train] DAYS_EMPLOYED anomalies replaced: 55,374 (18.01%)
  [test] DAYS_EMPLOYED anomalies replaced: 9,274 (19.03%)

  DAYS_EMPLOYED after fix:
  min=-17912  max=0  null%=18.01%
  DAYS_EMPLOYED_ANOM counts: {0: 252137, 1: 55374}


In [4]:
if 'previous_application' in dfs:
    pa = dfs['previous_application'].copy()
    pa_sentinel_cols = [
        'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT',
        'AMT_GOODS_PRICE', 'SELLERPLACE_AREA', 'CNT_PAYMENT',
        'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
        'DAYS_LAST_DUE', 'DAYS_TERMINATION',
    ]
    pa_sentinel_cols = [c for c in pa_sentinel_cols if c in pa.columns]
    for col in pa_sentinel_cols:
        n = (pa[col] == 365243).sum()
        if n > 0:
            pa[col] = pa[col].replace(365243, np.nan)
            print(f'  previous_application [{col}] — replaced {n:,} sentinel values')
    dfs['previous_application'] = pa
    print(f'\n  previous_application shape after fix: {pa.shape}')

  previous_application [DAYS_FIRST_DRAWING] — replaced 934,444 sentinel values
  previous_application [DAYS_FIRST_DUE] — replaced 40,645 sentinel values
  previous_application [DAYS_LAST_DUE_1ST_VERSION] — replaced 93,864 sentinel values
  previous_application [DAYS_LAST_DUE] — replaced 211,221 sentinel values
  previous_application [DAYS_TERMINATION] — replaced 225,913 sentinel values

  previous_application shape after fix: (1670214, 37)


## 4. Create Missingness Flags (40–70% null columns)

When 40–70% of a column is missing, **the fact that it is missing** is itself informative.
Create `{col}_IS_NULL` binary flag **before** imputing — so the model learns from absence of data.

This is a standard technique in credit risk modeling at financial institutions.

In [5]:
NULL_THRESHOLD_FLAG = 0.40

null_pct_now = train.isnull().mean()
flag_cols = [c for c in null_pct_now[
    (null_pct_now > NULL_THRESHOLD_FLAG) & (null_pct_now <= NULL_THRESHOLD_DROP)
].index if c not in PROTECTED]

print(f'Columns to create IS_NULL flags for (40–70% null): {len(flag_cols)}')

flag_log = []
for col in flag_cols:
    flag_name = f'{col}_IS_NULL'
    train[flag_name] = train[col].isnull().astype(np.int8)
    test[flag_name]  = test[col].isnull().astype(np.int8) if col in test.columns else 0

    dr_null    = train.loc[train[flag_name]==1, 'TARGET'].mean()
    dr_notnull = train.loc[train[flag_name]==0, 'TARGET'].mean()
    signal     = abs(dr_null - dr_notnull) if pd.notna(dr_null) else 0

    flag_log.append({'Column': col, 'Null %': round(null_pct_now[col]*100, 1),
                     'Default rate NULL': round(dr_null, 4) if pd.notna(dr_null) else None,
                     'Default rate NOTNA': round(dr_notnull, 4),
                     'Signal (diff)': round(signal, 4)})
    print(f'  ✓ {flag_name:<50} signal={signal:.4f}')

flag_df = pd.DataFrame(flag_log).sort_values('Signal (diff)', ascending=False)
print(f'\nTop flag features by default-rate signal:')
display(flag_df.head(10))

Columns to create IS_NULL flags for (40–70% null): 21
  ✓ EXT_SOURCE_1_IS_NULL                               signal=0.0102
  ✓ APARTMENTS_AVG_IS_NULL                             signal=0.0219
  ✓ BASEMENTAREA_AVG_IS_NULL                           signal=0.0202
  ✓ YEARS_BEGINEXPLUATATION_AVG_IS_NULL                signal=0.0221
  ✓ FLOORSMAX_AVG_IS_NULL                              signal=0.0223
  ✓ LIVINGAREA_AVG_IS_NULL                             signal=0.0215
  ✓ NONLIVINGAREA_AVG_IS_NULL                          signal=0.0215
  ✓ APARTMENTS_MODE_IS_NULL                            signal=0.0219
  ✓ BASEMENTAREA_MODE_IS_NULL                          signal=0.0202
  ✓ YEARS_BEGINEXPLUATATION_MODE_IS_NULL               signal=0.0221
  ✓ FLOORSMAX_MODE_IS_NULL                             signal=0.0223
  ✓ LIVINGAREA_MODE_IS_NULL                            signal=0.0215
  ✓ APARTMENTS_MEDI_IS_NULL                            signal=0.0219
  ✓ BASEMENTAREA_MEDI_IS_NULL                    

,Column,Null %,Default rate NULL,Default rate NOTNA,Signal (diff)
20,EMERGENCYSTATE_MODE,47.4000,0.0926,0.0700,0.0226
18,TOTALAREA_MODE,48.3000,0.0923,0.0699,0.0224
10,FLOORSMAX_MODE,49.8000,0.0919,0.0697,0.0223
15,FLOORSMAX_MEDI,49.8000,0.0919,0.0697,0.0223
4,FLOORSMAX_AVG,49.8000,0.0919,0.0697,0.0223
14,YEARS_BEGINEXPLUATATION_MEDI,48.8000,0.0920,0.0699,0.0221
9,YEARS_BEGINEXPLUATATION_MODE,48.8000,0.0920,0.0699,0.0221
3,YEARS_BEGINEXPLUATATION_AVG,48.8000,0.0920,0.0699,0.0221
1,APARTMENTS_AVG,50.7000,0.0915,0.0696,0.0219
7,APARTMENTS_MODE,50.7000,0.0915,0.0696,0.0219


## 5. Impute Missing Values

- Numeric → **median** (robust to outliers)
- Categorical → **mode** (most frequent)

> **Leakage rule:** Fit statistics on **train only**. Apply same values to test.

In [6]:
num_cols = [c for c in train.select_dtypes(include='number').columns
            if c not in ID_COLS + ['TARGET']]
cat_cols = [c for c in train.select_dtypes(include='object').columns
            if c not in ID_COLS]

print(f'Numeric columns to impute   : {len(num_cols)}')
print(f'Categorical columns to impute: {len(cat_cols)}')

num_medians, cat_modes = {}, {}

for col in num_cols:
    if train[col].isnull().any():
        med = train[col].median()
        num_medians[col] = med
        train[col] = train[col].fillna(med)
        if col in test.columns:
            test[col] = test[col].fillna(med)

for col in cat_cols:
    if train[col].isnull().any():
        mode_val = train[col].mode()[0]
        cat_modes[col] = mode_val
        train[col] = train[col].fillna(mode_val)
        if col in test.columns:
            test[col] = test[col].fillna(mode_val)

remaining_train = train.drop(columns=['TARGET']).isnull().sum().sum()
remaining_test  = test.isnull().sum().sum()
print(f'\nRemaining nulls — train: {remaining_train} | test: {remaining_test}')
print(f'Imputation stats: {len(num_medians)} numeric medians, {len(cat_modes)} cat modes')

# Save imputation lookup
imp_log = pd.DataFrame(
    [{'Column': k, 'Type': 'numeric',     'Method': 'median', 'Value': v} for k,v in num_medians.items()] +
    [{'Column': k, 'Type': 'categorical', 'Method': 'mode',   'Value': v} for k,v in cat_modes.items()]
)
imp_log.to_csv(os.path.join(OUTPUT_DIR, 'imputation_lookup.csv'), index=False)
print('Saved: imputation_lookup.csv')

Numeric columns to impute   : 99
Categorical columns to impute: 15

Remaining nulls — train: 0 | test: 0
Imputation stats: 35 numeric medians, 5 cat modes
Saved: imputation_lookup.csv


## 6. Label Encode Categorical Columns

Tree-based models (LightGBM, XGBoost) only need Label Encoding — one-hot would explode dimensionality.

> **Rule:** Fit on combined train+test values to avoid unseen-label errors at inference.

In [7]:
cat_cols = [c for c in train.select_dtypes('object').columns if c not in ID_COLS]
le_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([
        train[col].astype(str),
        test[col].astype(str) if col in test.columns else pd.Series(dtype=str)
    ]).unique()
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    if col in test.columns:
        test[col] = le.transform(test[col].astype(str))
    le_encoders[col] = le
    print(f'  ✓ {col:<45} classes={len(le.classes_)}')

print(f'\n{len(cat_cols)} categorical columns label-encoded.')

  ✓ NAME_CONTRACT_TYPE                            classes=2
  ✓ CODE_GENDER                                   classes=3
  ✓ FLAG_OWN_CAR                                  classes=2
  ✓ FLAG_OWN_REALTY                               classes=2
  ✓ NAME_TYPE_SUITE                               classes=7
  ✓ NAME_INCOME_TYPE                              classes=8
  ✓ NAME_EDUCATION_TYPE                           classes=5
  ✓ NAME_FAMILY_STATUS                            classes=6
  ✓ NAME_HOUSING_TYPE                             classes=6
  ✓ OCCUPATION_TYPE                               classes=18
  ✓ WEEKDAY_APPR_PROCESS_START                    classes=7
  ✓ ORGANIZATION_TYPE                             classes=58
  ✓ HOUSETYPE_MODE                                classes=3
  ✓ WALLSMATERIAL_MODE                            classes=7
  ✓ EMERGENCYSTATE_MODE                           classes=2

15 categorical columns label-encoded.


## 7. Cap Outliers at 99th Percentile

Financial outliers are often real (genuinely high incomes, large credits) — **cap rather than drop**.
Cap thresholds computed on train only, applied to both.

In [8]:
OUTLIER_CAP_COLS = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'REGION_POPULATION_RELATIVE',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'AMT_REQ_CREDIT_BUREAU_YEAR', 'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_HOUR',
]
OUTLIER_CAP_COLS = [c for c in OUTLIER_CAP_COLS if c in train.columns]

cap_log = []
for col in OUTLIER_CAP_COLS:
    q99 = train[col].quantile(0.99)
    n_tr = (train[col] > q99).sum()
    n_te = (test[col] > q99).sum() if col in test.columns else 0
    train[col] = train[col].clip(upper=q99)
    if col in test.columns:
        test[col] = test[col].clip(upper=q99)
    cap_log.append({'Column': col, 'Cap (q99)': round(q99, 2),
                    'Capped in train': n_tr, 'Capped in test': n_te})

# Hard cap: CNT_CHILDREN > 10 are data errors
if 'CNT_CHILDREN' in train.columns:
    train['CNT_CHILDREN'] = train['CNT_CHILDREN'].clip(upper=10)
    test['CNT_CHILDREN']  = test['CNT_CHILDREN'].clip(upper=10)

cap_df = pd.DataFrame(cap_log)
display(cap_df)
print('Outlier capping complete.')

,Column,Cap (q99),Capped in train,Capped in test
0,AMT_INCOME_TOTAL,"472,500.0000",3014,574
1,AMT_CREDIT,"1,854,000.0000",3075,495
2,AMT_ANNUITY,"70,006.5000",3070,844
3,AMT_GOODS_PRICE,"1,800,000.0000",1431,2
4,CNT_CHILDREN,3.0000,555,71
5,CNT_FAM_MEMBERS,5.0000,529,64
6,REGION_POPULATION_RELATIVE,0.0700,0,0
7,OBS_30_CNT_SOCIAL_CIRCLE,10.0000,2782,458
8,DEF_30_CNT_SOCIAL_CIRCLE,2.0000,1515,243
9,OBS_60_CNT_SOCIAL_CIRCLE,10.0000,2691,450


Outlier capping complete.


## 8. Log1p Transform Skewed Columns

Columns with |skew| > 1.5 get `np.log1p` applied — compresses the right tail.
Benefits PCA and any linear components in the ensemble.

> LightGBM is invariant to monotonic transforms, so this is safe for tree models too.

In [11]:
# ── Log1p transform — only where it genuinely helps ──────────────────────────
# Note: XGBoost/LightGBM are invariant to this transform.
# We apply it ONLY for columns that will feed into PCA in Phase 5.
# Skew in binary cols and left-skewed cols is expected and harmless.

num_cols_now = [
    c for c in train.select_dtypes(include='number').columns
    if c not in ID_COLS + ['TARGET']
    and not c.endswith('_IS_NULL')
    and not c.endswith('_ANOM')
    and 'DAYS' not in c
    and 'FLAG' not in c
]

SKEW_THRESHOLD = 1.5

# Skip columns that are binary (only 2 unique values) — skew is meaningless there
binary_cols = [c for c in num_cols_now if train[c].nunique() <= 2]
candidates  = [
    c for c in num_cols_now
    if c not in binary_cols
    and train[c].skew() > SKEW_THRESHOLD
    and train[c].min() >= 0
]

log_cols  = []
skip_cols = []

for col in candidates:
    skew_before  = train[col].skew()
    transformed  = np.log1p(train[col])
    skew_after   = transformed.skew()

    if abs(skew_after) < abs(skew_before):
        train[col] = transformed
        if col in test.columns:
            test[col] = np.log1p(test[col])
        log_cols.append(col)
    else:
        skip_cols.append(col)

print(f'Binary cols skipped (skew meaningless) : {len(binary_cols)}')
print(f'Log1p applied (genuinely improved)     : {len(log_cols)}')
print(f'Skipped (log1p made worse)             : {len(skip_cols)}')
print(f'\nTransformed cols saved → LOG_TRANSFORMED_COLS')
print(f'Phase 5 (PCA) will use these {len(log_cols)} columns in their transformed state.')
print(f'\nFor XGBoost/LightGBM: skew does not affect tree models — proceed to Phase 4.')

# Save the list for Phase 5
LOG_TRANSFORMED_COLS = log_cols

Binary cols skipped (skew meaningless) : 11
Log1p applied (genuinely improved)     : 20
Skipped (log1p made worse)             : 0

Transformed cols saved → LOG_TRANSFORMED_COLS
Phase 5 (PCA) will use these 20 columns in their transformed state.

For XGBoost/LightGBM: skew does not affect tree models — proceed to Phase 4.


In [16]:
# Save log-transformed column list for Phase 5 PCA
with open(os.path.join(OUTPUT_DIR, 'log_transformed_cols.txt'), 'w') as f:
    f.write('\n'.join(LOG_TRANSFORMED_COLS))
print(f'Saved log_transformed_cols.txt — {len(LOG_TRANSFORMED_COLS)} columns')

Saved log_transformed_cols.txt — 20 columns


## 9. Preprocess Secondary Files

Each secondary file: drop high-null cols → fix 365243 → impute → label encode.
We do **not** merge yet — aggregation happens in Phase 4.

In [12]:
def preprocess_secondary(df, name, null_drop_thresh=0.60):
    df = df.copy()
    ID_KEYS = ['SK_ID_CURR', 'SK_ID_BUREAU', 'SK_ID_PREV']

    # 1. Drop extreme null cols
    null_pct  = df.isnull().mean()
    drop_cols = [c for c in null_pct[null_pct > null_drop_thresh].index if c not in ID_KEYS]
    df = df.drop(columns=drop_cols, errors='ignore')
    if drop_cols:
        print(f'  [{name}] Dropped {len(drop_cols)} high-null cols: {drop_cols}')

    # 2. Fix 365243 sentinel in all numeric cols
    for col in df.select_dtypes(include='number').columns:
        n = (df[col] == 365243).sum()
        if n > 0:
            df[col] = df[col].replace(365243, np.nan)
            print(f'  [{name}] {col}: replaced {n:,} sentinel values')

    # 3. Median impute numeric
    for col in df.select_dtypes(include='number').columns:
        if col not in ID_KEYS and df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    # 4. Mode impute + label encode categoricals
    for col in df.select_dtypes(include='object').columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode()[0])
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    remaining = df.isnull().sum().sum()
    print(f'  [{name}] Final shape: {df.shape} | Remaining nulls: {remaining}')
    return df


SECONDARY_NAMES = [
    'bureau', 'bureau_balance', 'previous_application',
    'POS_CASH_balance', 'installments_payments', 'credit_card_balance'
]
cleaned = {}
for name in SECONDARY_NAMES:
    if name in dfs:
        print(f'\nProcessing: {name}')
        cleaned[name] = preprocess_secondary(dfs[name], name)
print('\nAll secondary files preprocessed.')


Processing: bureau
  [bureau] Dropped 2 high-null cols: ['AMT_CREDIT_MAX_OVERDUE', 'AMT_ANNUITY']
  [bureau] SK_ID_CURR: replaced 8 sentinel values
  [bureau] Final shape: (1716428, 15) | Remaining nulls: 8

Processing: bureau_balance
  [bureau_balance] Final shape: (27299925, 3) | Remaining nulls: 0

Processing: previous_application
  [previous_application] Dropped 3 high-null cols: ['RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'DAYS_FIRST_DRAWING']
  [previous_application] SK_ID_CURR: replaced 1 sentinel values
  [previous_application] Final shape: (1670214, 34) | Remaining nulls: 1

Processing: POS_CASH_balance
  [POS_CASH_balance] SK_ID_CURR: replaced 14 sentinel values
  [POS_CASH_balance] Final shape: (10001358, 8) | Remaining nulls: 14

Processing: installments_payments
  [installments_payments] SK_ID_CURR: replaced 12 sentinel values
  [installments_payments] Final shape: (13605401, 8) | Remaining nulls: 12

Processing: credit_card_balance
  [credit_card_balance] Final

## 10. Schema Alignment — Train & Test

After all transforms, train and test must have identical columns (except TARGET).
Column mismatch causes runtime errors at inference.

In [13]:
train_cols = set(train.drop(columns=['TARGET']).columns)
test_cols  = set(test.columns)

in_train_not_test = train_cols - test_cols
in_test_not_train = test_cols  - train_cols

print(f'Train cols (excl TARGET): {len(train_cols)}')
print(f'Test cols               : {len(test_cols)}')

if in_train_not_test:
    print(f'\n  In train NOT test ({len(in_train_not_test)}): {in_train_not_test}')
    for col in in_train_not_test:
        test[col] = 0
    print('  Added missing cols to test, filled with 0.')

if in_test_not_train:
    print(f'\n  In test NOT train ({len(in_test_not_train)}): {in_test_not_train}')
    test.drop(columns=list(in_test_not_train), inplace=True, errors='ignore')
    print('  Dropped extra cols from test.')

feature_cols = [c for c in train.columns if c not in ['TARGET', 'SK_ID_CURR']]
test = test[['SK_ID_CURR'] + [c for c in feature_cols if c in test.columns]]

print(f'\nFinal — Train: {train.shape} | Test: {test.shape}')
print('Schema aligned.')

Train cols (excl TARGET): 115
Test cols               : 115

Final — Train: (307511, 116) | Test: (48744, 115)
Schema aligned.


In [14]:
train.to_csv(os.path.join(OUTPUT_DIR, 'train_clean.csv'), index=False)
test.to_csv(os.path.join(OUTPUT_DIR,  'test_clean.csv'),  index=False)
print(f'Saved train_clean.csv  -> {train.shape}')
print(f'Saved test_clean.csv   -> {test.shape}')

for name, df in cleaned.items():
    df.to_csv(os.path.join(OUTPUT_DIR, f'{name}_clean.csv'), index=False)
    print(f'Saved {name}_clean.csv  -> {df.shape}')

print(f'\nAll files saved to: {os.path.abspath(OUTPUT_DIR)}')

Saved train_clean.csv  -> (307511, 116)
Saved test_clean.csv   -> (48744, 115)
Saved bureau_clean.csv  -> (1716428, 15)
Saved bureau_balance_clean.csv  -> (27299925, 3)
Saved previous_application_clean.csv  -> (1670214, 34)
Saved POS_CASH_balance_clean.csv  -> (10001358, 8)
Saved installments_payments_clean.csv  -> (13605401, 8)
Saved credit_card_balance_clean.csv  -> (3840312, 23)

All files saved to: c:\Users\User\Desktop\project\new notebooks\preprocessed


In [15]:
print('=' * 65)
print('  PHASE 3 PREPROCESSING QUALITY REPORT')
print('=' * 65)

report = {
    'Columns dropped (>70% null)'       : len(drop_cols),
    'IS_NULL flags created'             : len(flag_cols),
    'DAYS_EMPLOYED anomalies fixed'     : int(train['DAYS_EMPLOYED_ANOM'].sum()),
    'Numeric columns imputed (median)'  : len(num_medians),
    'Categorical columns encoded (LE)'  : len(le_encoders),
    'Columns capped at q99'             : len(cap_df),
    'Columns log1p transformed'         : len(log_cols),
    'Final train shape'                 : str(train.shape),
    'Final test shape'                  : str(test.shape),
    'Remaining nulls in train'          : int(train.isnull().sum().sum()),
    'Remaining nulls in test'           : int(test.isnull().sum().sum()),
}
for k, v in report.items():
    print(f'  {k:<45} : {v}')

print()
print('DATA LEAKAGE CHECKLIST:')
checks = [
    'Medians computed on train only — applied to test',
    'LabelEncoders fitted on train+test combined values',
    'Cap thresholds from train quantiles only',
    'Log1p applied identically to train and test',
    'TARGET column never used in any transformation',
    'Missingness flags created BEFORE imputation',
]
for c in checks:
    print(f'  [OK] {c}')

print()
print('Next -> Phase 4: Feature Engineering')

  PHASE 3 PREPROCESSING QUALITY REPORT
  Columns dropped (>70% null)                   : 28
  IS_NULL flags created                         : 21
  DAYS_EMPLOYED anomalies fixed                 : 55374
  Numeric columns imputed (median)              : 35
  Categorical columns encoded (LE)              : 15
  Columns capped at q99                         : 17
  Columns log1p transformed                     : 20
  Final train shape                             : (307511, 116)
  Final test shape                              : (48744, 115)
  Remaining nulls in train                      : 0
  Remaining nulls in test                       : 0

DATA LEAKAGE CHECKLIST:
  [OK] Medians computed on train only — applied to test
  [OK] LabelEncoders fitted on train+test combined values
  [OK] Cap thresholds from train quantiles only
  [OK] Log1p applied identically to train and test
  [OK] TARGET column never used in any transformation
  [OK] Missingness flags created BEFORE imputation

Next -> Phas